# FPW Water Source Classifier â€” Phase 1
Classify free-text `planSource` strings from fracking completion reports into source type buckets,
then assess coverage by record count and reported volume.

In [ ]:
import pandas as pd
import re

jct = pd.read_parquet('data/well_junction_table.parquet')
master = pd.read_parquet('data/FPW_master_water_source.parquet')

print('Junction table:', jct.shape)
print('Master sources:', master.shape)

In [ ]:
SKINNY_PATH = r'G:\My Drive\production\repos\openFF_data_2026_04_03\skinny_df.parquet'

# One lat/lon per well â€” all rows for a given api10 share the same coordinates
well_coords = (
    pd.read_parquet(SKINNY_PATH, columns=['api10', 'bgLatitude', 'bgLongitude'])
    .groupby('api10', as_index=False)
    .first()
    .rename(columns={'bgLatitude': 'well_lat', 'bgLongitude': 'well_lon'})
)

jct = jct.merge(well_coords, on='api10', how='left')

has_coords = jct['well_lat'].notna().sum()
print(f'Junction rows with well coordinates: {has_coords:,} / {len(jct):,} '
      f'({round(has_coords/len(jct)*100, 1)}%)')

## Classifier

Rules are applied in priority order â€” first match wins.

| Type | Meaning |
|---|---|
| `reuse` | Recycled/produced water, rainwater |
| `interconnection` | Public water system tap, hydrant, vault, authority, etc. |
| `groundwater` | Wells, springs, aquifers |
| `impoundment` | Constructed storage pond/pit (surface water stored) |
| `surface_direct` | Named natural water body (creek, river, lake, etc.) |
| `srbc_only` | Has SRBC docket number but type unclear from name |
| `ambiguous` | No pattern matched |

In [ ]:
# Each entry: (type_label, regex_pattern)
# Case-insensitive; first match wins
RULES = [
    ('reuse',         r'recycl|reuse|re-use|flowback|produced water|rain\s*water'),
    ('interconnection', r'\bintc\b|\btap\b|\bvending\b|\bvault\b|\bhydrant\b'
                        r'|\bauthority\b|\bmunicipal\b|water\s+co[^u]|\bmeter\b'
                        r'|public\s+water|\bpwsid\b|water\s+system|water\s+service'
                        r'|water\s+company|interconnect'),
    ('groundwater',   r'\bwell\b|\bspring\b|\baquifer\b|ground\s*water'),
    ('impoundment',   r'impoundment|\bpit\b|frac\s+pond'),
    ('surface_direct',r'creek|river|\brun\b|stream|\blake\b|\bpond\b|reservoir'
                      r'|brook|\bbranch\b|\bfork\b|hollow|\bdam\b|hatchery'),
    ('srbc_only',     r'srbc\s+docket'),
    ('dont_know',     r'\bSWW\b|\bSPWA\b|\bSWPA\b|\bNKWA\b|\bMAWC\b|\bMANK\b'
                      r'|\bAqua\b|\bWI\b|\bbrine\b|quarry|limestone\s+mine'
                      r'|\bClermont\b|\bAWS\b'),
]
def classify_source(name):
    if pd.isna(name):
        return 'no_source'
    s = str(name)
    for label, pattern in RULES:
        if re.search(pattern, s, re.IGNORECASE):
            return label
    return 'ambiguous'

jct['source_type'] = jct['planSource'].apply(classify_source)

# Also flag SRBC presence as a separate boolean
jct['has_srbc'] = jct['planSource'].str.contains(r'srbc\s+docket', case=False, na=False)

print(jct['source_type'].value_counts())

## Coverage by record count and volume

In [ ]:
summary = (
    jct.groupby('source_type')
    .agg(
        records=('planSource', 'count'),
        volume_gal=('volume', 'sum'),
        unique_sources=('site_clean', 'nunique'),
    )
    .assign(
        pct_records=lambda d: (d['records'] / d['records'].sum() * 100).round(1),
        pct_volume=lambda d: (d['volume_gal'] / d['volume_gal'].sum() * 100).round(1),
        volume_Mgal=lambda d: (d['volume_gal'] / 1e6).round(1),
    )
    .sort_values('volume_gal', ascending=False)
    [['records', 'pct_records', 'volume_Mgal', 'pct_volume', 'unique_sources']]
)
print(summary.to_string())

## Breakdown by operator

In [ ]:
op_pivot = (
    jct.groupby(['operator_clean', 'source_type'])['volume']
    .sum()
    .unstack(fill_value=0)
    .assign(total=lambda d: d.sum(axis=1))
    .sort_values('total', ascending=False)
)
# Show as % of each operator's total volume
op_pct = op_pivot.div(op_pivot['total'], axis=0).mul(100).round(1)
op_pct['total_Mgal'] = (op_pivot['total'] / 1e6).round(1)

print('Top 20 operators by total reported volume (% by source type):')
print(op_pct.sort_values('total_Mgal', ascending=False).head(20).to_string())

## Sample strings per type â€” for classifier validation

In [ ]:
for stype in jct['source_type'].unique():
    samples = (
        jct.loc[jct['source_type'] == stype, 'planSource']
        .dropna()
        .value_counts()
        .head(8)
        .index.tolist()
    )
    print(f'\n--- {stype} ---')
    for s in samples:
        print(' ', s)

## Ambiguous strings â€” candidates for rule refinement

In [ ]:
ambig = (
    jct.loc[jct['source_type'] == 'ambiguous']
    .groupby('planSource')['volume']
    .agg(['count', 'sum'])
    .rename(columns={'count': 'records', 'sum': 'volume_gal'})
    .sort_values('volume_gal', ascending=False)
)
print(f'Ambiguous: {len(ambig)} unique strings')
print(ambig.head(30).to_string())

## NHD linkage candidate pool
Surface, impoundment, and SRBC-only sources â€” these are candidates for NHD matching.

In [ ]:
nhd_candidates = jct[
    jct['source_type'].isin(['surface_direct', 'impoundment', 'srbc_only'])
].copy()

print('NHD linkage candidate records:', len(nhd_candidates))
print('With SRBC docket:', nhd_candidates['has_srbc'].sum())
print('Volume (Mgal):', round(nhd_candidates['volume'].sum() / 1e6, 1))
print()

# Coordinate coverage â€” source coords (master) vs well proxy coords (skinny_df)
cands_have_src_coord  = nhd_candidates['site_ID'].isin(
    master.loc[master['Latitude'].notna(), 'site_ID'])
cands_have_well_coord = nhd_candidates['well_lat'].notna()

print(f'Source coordinates (master table):  {cands_have_src_coord.sum():,} records '
      f'({round(cands_have_src_coord.mean()*100,1)}%)')
print(f'Well proxy coordinates (skinny_df): {cands_have_well_coord.sum():,} records '
      f'({round(cands_have_well_coord.mean()*100,1)}%)')
print(f'Either coordinate available:        '
      f'{(cands_have_src_coord | cands_have_well_coord).sum():,} records '
      f'({round((cands_have_src_coord | cands_have_well_coord).mean()*100,1)}%)')
print()

# Unique sources â€” deduplicate by site_ID where real, else by planSource
valid_id = nhd_candidates['site_ID'].notna() & ~nhd_candidates['site_ID'].isin(['', 'ambiguous'])
uniq_with_id = (nhd_candidates[valid_id]
                .drop_duplicates('site_ID')
                [['site_ID', 'planSource', 'site_clean', 'source_type', 'has_srbc']])
uniq_no_id   = (nhd_candidates[~valid_id]
                .drop_duplicates('planSource')
                [['site_ID', 'planSource', 'site_clean', 'source_type', 'has_srbc']])
candidates_unique = pd.concat([uniq_with_id, uniq_no_id], ignore_index=True)

# Attach best available coordinate: source coord preferred, well proxy as fallback
candidates_unique = candidates_unique.merge(
    master[['site_ID', 'site_name', 'Latitude', 'Longitude']], on='site_ID', how='left')

well_proxy = (nhd_candidates
              .groupby('planSource')[['well_lat', 'well_lon']]
              .median()
              .reset_index()
              .rename(columns={'well_lat': 'proxy_lat', 'well_lon': 'proxy_lon'}))
candidates_unique = candidates_unique.merge(well_proxy, on='planSource', how='left')

candidates_unique['coord_lat'] = candidates_unique['Latitude'].fillna(candidates_unique['proxy_lat'])
candidates_unique['coord_lon'] = candidates_unique['Longitude'].fillna(candidates_unique['proxy_lon'])
candidates_unique['coord_source'] = candidates_unique['Latitude'].notna().map(
    {True: 'source', False: 'well_proxy'})

any_coord = candidates_unique['coord_lat'].notna().sum()
print(f'Unique candidate sources:           {len(candidates_unique)}')
print(f'With any coordinate:                {any_coord} ({round(any_coord/len(candidates_unique)*100,1)}%)')
print()
print(candidates_unique[['planSource', 'source_type', 'has_srbc',
                          'coord_lat', 'coord_lon', 'coord_source']]
      .sort_values('coord_lat', na_position='last').head(20).to_string())

## Extract water feature names from SRBC strings
For NHD matching we need a clean water body name. SRBC strings embed the feature name
alongside operator and site names â€” this extractor isolates it.

In [ ]:
WATER_TERMS = (
    r'\b(?:creek|river|run|stream|lake|pond|reservoir|brook|branch|'
    r'fork|hollow|dam|hatchery|spring|susquehanna|wyalusing)\b'
)

# Known operator/site tokens that appear as leading noise in no-comma strings
_OP_PREFIXES = {
    'cabot', 'coterra', 'bkv', 'sfgs', 'sgfs', 'bnr', 'carrizo',
    'repsol', 'chesapeake', 'eqt', 'range', 'swn', 'talisman',
    'cnx', 'well', 'field', 'diaz', 'pro-environmental',
    'garrison',  # BKV site name that leaks in
}

def _strip_leading_noise(feature):
    """Remove leading operator tokens and common prefixes from feature name."""
    # Remove list-number prefix ("1. ", "6 ") and "X% Freshwater from" preamble
    feature = re.sub(r'^\d+[\.\s]\s*', '', feature)
    feature = re.sub(r'^\d+%\s*freshwater\s+from\s+', '', feature, flags=re.IGNORECASE)
    # Strip leading tokens that are known operator abbreviations/site names
    words = feature.split()
    while words:
        token = re.sub(r'[\-\.\#\d]', '', words[0]).lower()
        if token in _OP_PREFIXES or token == '':
            words = words[1:]
        else:
            break
    return ' '.join(words).strip(' -.')


def extract_srbc_feature(row):
    """Return (feature_name, docket_number) from an SRBC junction-table row."""
    s = row['planSource']
    if pd.isna(s):
        return None, None
    s = str(s)

    # Pull docket number â€” handles both "Number" and "No." variants
    m = re.search(r'SRBC\s+Docket\s+(?:No\.|Number)?\s*(\d[\d\-]*)', s, re.IGNORECASE)
    docket = m.group(1) if m else None

    # Strip SRBC references:
    #   [SRBC...]  â€” normal bracketed form
    #   (SRBC...]  â€” malformed: opens with ( but closes with ]
    #   bare "SRBC Docket No. XXXXXX" â€” no brackets at all
    s_clean = re.sub(r'[\[\(]SRBC[^\]\)]*[\]\)]', '', s, flags=re.IGNORECASE)
    s_clean = re.sub(r'\bSRBC\s+Docket\s+(?:No\.|Number)?\s*\d[\d\-]*', '', s_clean,
                     flags=re.IGNORECASE)
    s_clean = s_clean.strip()

    # Pass 1: search comma-separated segments (parens removed)
    # Water feature is often last segment, but not always
    s_no_parens = re.sub(r'\([^)]*\)', '', s_clean)
    segments = [seg.strip() for seg in s_no_parens.split(',')]
    feature = None
    for seg in reversed(segments):
        if re.search(WATER_TERMS, seg, re.IGNORECASE):
            feature = seg.strip(' -.')
            break

    # Pass 2: water feature buried in a parenthetical, e.g. "(Meshoppen Creek)"
    if feature is None:
        for paren in re.findall(r'\(([^)]+)\)', s_clean):
            if re.search(WATER_TERMS, paren, re.IGNORECASE):
                feature = paren.strip()
                break

    if feature is None:
        return None, docket

    # Truncate at end of the last water keyword â€” removes trailing metadata
    # e.g. "Fall Brook Creek DCNR 587" -> "Fall Brook Creek"
    m_last = None
    for m in re.finditer(WATER_TERMS, feature, re.IGNORECASE):
        m_last = m
    if m_last:
        feature = feature[:m_last.end()].strip(' -.')

    # Remove leading noise (operator names, list prefixes, "X% freshwater from")
    feature = _strip_leading_noise(feature)

    return feature if feature else None, docket


# Apply to all SRBC-containing rows
srbc_mask = jct['planSource'].str.contains(r'srbc\s+docket', case=False, na=False)
srbc_rows = jct[srbc_mask].copy()

results = srbc_rows.apply(extract_srbc_feature, axis=1)
srbc_rows['feature_name'] = results.apply(lambda x: x[0])
srbc_rows['srbc_docket'] = results.apply(lambda x: x[1])

total = len(srbc_rows)
matched = srbc_rows['feature_name'].notna().sum()
print(f'SRBC records:   {total}')
print(f'Feature found:  {matched}  ({round(matched/total*100,1)}%)')
print(f'No match:       {total - matched}')

In [ ]:
# Unique extracted feature names by frequency
print('=== Extracted feature names (top 40 by record count) ===')
print(srbc_rows.groupby('feature_name')['planSource'].count()
      .sort_values(ascending=False).head(40).to_string())

print()
print('=== Unmatched planSource strings ===')
unmatched = (srbc_rows[srbc_rows['feature_name'].isna()]['planSource']
             .value_counts())
print(unmatched.to_string())